# Adversarial Search: Solving Tic-Tac-Toe with Heuristic Alpha-Beta Tree Search

## Introduction 

Multiplayer games can be implemented as:
1. Nondeterministic actions: The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. __Heuristic Alpha-Beta Tree Search:__ Cut off tree search and use heuristic to estimate state value. 
4. Monte Carlo Tree search: Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)). The game is a __zero-sum game__: Win by x results in +1, win by o in -1 and a tie has a value of 0. Max plays x and tries to maximize the outcome while Min plays o and tries to minimize the outcome.   

We will implement
* Heuristic Alpha-Beta Tree Search

The algorithms search the game tree and we could return a conditional plan (or partial plan if cut offs are used), but the implementation here only identifies and returns the optimal next move.

The code for the basic functions used for search are implemented in [tictactoe.py](tictactoe.py).

In [1]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("tictactoe.py", ".")

In [2]:
from tictactoe import empty_board, actions, result, terminal, utility, other, show_board
from tictactoe import random_player, play 

# Heuristic Alpha-Beta Tree Search

See AIMA page 156ff. 


## Heuristic Evaluation Function

In [3]:
import numpy as np

def eval_fun(state, player = 'x'):
    """heuristic for utility of state. Returns score for a node:
    1. For terminal states it returns the utility. 
    2. For non-terminal states, it calculates a weighted linear function using features of the state. 
    The features we look at are 2 in a row/col/diagonal where the 3rd square is empty. We assume that
    the more of these positions we have, the higher the chance of winning.
    We need to be careful that the utility of the heuristic stays between [-1,1]. 
    Note that the largest possible number of these positions is 2. I weigh the count by 0.4, 
    guaranteeing that is in the needed range.
    
    Function Returns: heuristic value, terminal?"""
    
    # terminal state?
    u = utility(state, player)
    if u is not None: return u, True
      
    score = 0
    board = np.array(state).reshape((3,3))
    diagonals = np.array([[board[i][i] for i in range(len(board))], 
                          [board[i][len(board)-i-1] for i in range(len(board))]])
    
    for a_board in [board, np.transpose(board), diagonals]:
        for row in a_board:
            if sum(row == player) == 2 and any(row ==' '): score += .4
            if sum(row == other(player)) == 2 and any(row ==' '): score -= .4
    
    return score, False

In [4]:
board = empty_board() 
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'x'
board[2] = 'x' 
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'x'
board[3] = 'x' 
board[4] = 'o'
board[8] = 'o'
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'x' 
board[4] = 'o'
show_board(board)
print(f"eval for x: {eval_fun(board)}")
print(f"eval for o: {eval_fun(board, 'o')}")

eval for x: (0, False)
eval for o: (0, False)
eval for x: (1, True)
eval for o: (-1, True)
eval for x: (0.8, False)
eval for o: (-0.8, False)
eval for x: (0.0, False)
eval for o: (0.0, False)


## Search with Cutoff

We add a cutoff to the Recursive DFS algorithm for Minimax Search with Alpha-Beta Pruning (see AIMA page 156ff). We use the heuristic evaluation function and the back-up the value using minimax search with alpha-beta pruning to determine the next move.

In [5]:
import math

# global variables
DEBUG = 1 # 1 ... count nodes, 2 ... debug each node
COUNT = 0

def alpha_beta_search(board, cutoff = None, player = 'x'):
    """start the search. cutoff = None is minimax search with alpha-beta pruning."""
    global DEBUG, COUNT
    COUNT = 0

    value, move = max_value_ab(board, player, -math.inf, +math.inf, 0, cutoff)
    
    if DEBUG >= 1: print(f"Number of nodes searched (cutoff = {cutoff}): {COUNT}") 
    
    return {"move": move, "value": value}

def max_value_ab(state, player, alpha, beta, depth, cutoff):
    """player's best move."""
    global DEBUG, COUNT
    COUNT += 1
    
    # cut off and terminal test
    v, terminal = eval_fun(state, player)
    if((cutoff is not None and depth >= cutoff) or terminal): 
        if(terminal): 
            alpha, beta = v, v
        if DEBUG >= 2: print(f"stopped at {depth}: {state} term: {terminal} eval: {v} [{alpha}, {beta}]" ) 
        return v, None
    
    v, move = -math.inf, None

    # check all possible actions in the state, update alpha and return move with the largest value
    for a in actions(state):
        v2, a2 = min_value_ab(result(state, player, a), player, alpha, beta, depth + 1, cutoff)
        if v2 > v:
            v, move = v2, a
            alpha = max(alpha, v)
        if v >= beta: return v, move
    
    return v, move

def min_value_ab(state, player, alpha, beta, depth, cutoff):
    """opponent's best response."""
    global DEBUG, COUNT
    COUNT += 1
    
    # cut off and terminal test
    v, terminal = eval_fun(state, player)
    if((cutoff is not None and depth >= cutoff) or terminal): 
        if(terminal): 
            alpha, beta = v, v
        if DEBUG >= 2: print(f"stopped at {depth}: {state} term: {terminal} eval: {v} [{alpha}, {beta}]" ) 
        return v, None
    
    v, move = +math.inf, None

    # check all possible actions in the state, update beta and return move with the smallest value
    for a in actions(state):
        v2, a2 = max_value_ab(result(state, other(player), a), player, alpha, beta, depth + 1, cutoff)
        if v2 < v:
            v, move = v2, a
            beta = min(beta, v)
        if v <= alpha: return v, move
    
    return v, move

## Some Tests

### x is about to win (play 8)

In [6]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))

print()
%time display(alpha_beta_search(board, 4))

print()
%time display(alpha_beta_search(board))

Board:

Number of nodes searched (cutoff = 2): 13


{'move': 8, 'value': 1}

CPU times: user 2.68 ms, sys: 0 ns, total: 2.68 ms
Wall time: 2.57 ms

Number of nodes searched (cutoff = 4): 47


{'move': 2, 'value': 1}

CPU times: user 2.56 ms, sys: 875 μs, total: 3.43 ms
Wall time: 3.27 ms

Number of nodes searched (cutoff = None): 61


{'move': 2, 'value': 1}

CPU times: user 3.78 ms, sys: 0 ns, total: 3.78 ms
Wall time: 3.62 ms


### o is about to win

In [7]:
board = empty_board() 
board[0] = 'o'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'
board[8] = 'x'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board))

Board:

Number of nodes searched (cutoff = 2): 11


{'move': 2, 'value': -1}

CPU times: user 1.48 ms, sys: 43 μs, total: 1.52 ms
Wall time: 1.33 ms

Number of nodes searched (cutoff = None): 15


{'move': 2, 'value': -1}

CPU times: user 1.47 ms, sys: 0 ns, total: 1.47 ms
Wall time: 1.32 ms


### x can draw if it chooses 7

In [8]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
board[4] = 'o'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:

Number of nodes searched (cutoff = 2): 21


{'move': 7, 'value': -0.4}

CPU times: user 2.13 ms, sys: 10 μs, total: 2.14 ms
Wall time: 1.97 ms

Number of nodes searched (cutoff = 4): 81


{'move': 7, 'value': 0}

CPU times: user 5.15 ms, sys: 9 μs, total: 5.16 ms
Wall time: 5.04 ms

Number of nodes searched (cutoff = None): 101


{'move': 7, 'value': 0}

CPU times: user 4.67 ms, sys: 1.01 ms, total: 5.68 ms
Wall time: 5.56 ms


### Empty board: Only a draw an be guaranteed

In [9]:
board = empty_board() 

print("Board:")
show_board(board)


print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:

Number of nodes searched (cutoff = 2): 26


{'move': 0, 'value': 0}

CPU times: user 2.64 ms, sys: 10 μs, total: 2.65 ms
Wall time: 2.49 ms

Number of nodes searched (cutoff = 4): 541


{'move': 4, 'value': 0.0}

CPU times: user 31.6 ms, sys: 1.95 ms, total: 33.5 ms
Wall time: 33.3 ms



Number of nodes searched (cutoff = None): 18297


{'move': 0, 'value': 0}

CPU times: user 856 ms, sys: 666 μs, total: 857 ms
Wall time: 856 ms


### A bad situation

In [10]:
board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)

print()
%time display(alpha_beta_search(board, 2))
print()
%time display(alpha_beta_search(board, 4))
print()
%time display(alpha_beta_search(board))

Board:

Number of nodes searched (cutoff = 2): 29


{'move': 4, 'value': -0.8}

CPU times: user 794 μs, sys: 1.95 ms, total: 2.75 ms
Wall time: 2.56 ms

Number of nodes searched (cutoff = 4): 148


{'move': 1, 'value': -1}

CPU times: user 8.77 ms, sys: 0 ns, total: 8.77 ms
Wall time: 8.58 ms

Number of nodes searched (cutoff = None): 238


{'move': 1, 'value': -1}

CPU times: user 12.2 ms, sys: 8 μs, total: 12.3 ms
Wall time: 12.1 ms


## Experiments



### Minimax with Alpha-Beta Pruning vs. Random

In [11]:
def heuristic2_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = 2, player = player)["move"]

def heuristic4_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = 4, player = player)["move"]

def alpha_beta_player(board, player = 'x'):
    return alpha_beta_search(board, cutoff = None, player = player)["move"]

DEBUG = 1
print("heuristic2 vs. random:")
display(play(heuristic2_player, random_player, N = 3))

heuristic2 vs. random:
Number of nodes searched (cutoff = 2): 26
Number of nodes searched (cutoff = 2): 30
Number of nodes searched (cutoff = 2): 10
Number of nodes searched (cutoff = 2): 26
Number of nodes searched (cutoff = 2): 27
Number of nodes searched (cutoff = 2): 20
Number of nodes searched (cutoff = 2): 5
Number of nodes searched (cutoff = 2): 26
Number of nodes searched (cutoff = 2): 27
Number of nodes searched (cutoff = 2): 10


{'x': 3, 'o': 0, 'd': 0}

In [12]:
DEBUG = 0
print("heuristic2 vs. random:")
%time display(play(heuristic2_player, random_player))

print("heuristic4 vs. random:")
%time display(play(heuristic4_player, random_player))

print()
print("random vs. heuristic2")
%time display(play(random_player, heuristic2_player))

print("random vs. heuristic4")
%time display(play(random_player, heuristic4_player))

heuristic2 vs. random:


{'x': 96, 'o': 0, 'd': 4}

CPU times: user 416 ms, sys: 1.94 ms, total: 418 ms
Wall time: 416 ms
heuristic4 vs. random:


{'x': 99, 'o': 0, 'd': 1}

CPU times: user 5.29 s, sys: 0 ns, total: 5.29 s
Wall time: 5.29 s

random vs. heuristic2


{'x': 6, 'o': 76, 'd': 18}

CPU times: user 417 ms, sys: 2.66 ms, total: 420 ms
Wall time: 418 ms
random vs. heuristic4


{'x': 0, 'o': 87, 'd': 13}

CPU times: user 3.07 s, sys: 1.96 ms, total: 3.08 s
Wall time: 3.08 s


### Heuristic vs. Minimax with Alpha-Beta Pruning

In [13]:
DEBUG = 0

# Note: No randomness -> play only once

print("heuristic2 vs. alpha_beta")
%time display(play(heuristic2_player, alpha_beta_player, N = 1))

print()
print("alpha_beta vs. heuristic2")
%time display(play(alpha_beta_player, heuristic2_player, N = 1))

print()
print("heuristic4 vs alpha_beta")
%time display(play(heuristic4_player, alpha_beta_player, N = 1))

print()
print("alpha_beta vs. heuristic4")
%time display(play(alpha_beta_player, heuristic4_player, N = 1))

heuristic2 vs. alpha_beta


{'x': 0, 'o': 0, 'd': 1}

CPU times: user 119 ms, sys: 0 ns, total: 119 ms
Wall time: 119 ms

alpha_beta vs. heuristic2


{'x': 1, 'o': 0, 'd': 0}

CPU times: user 881 ms, sys: 0 ns, total: 881 ms
Wall time: 881 ms

heuristic4 vs alpha_beta


{'x': 0, 'o': 0, 'd': 1}

CPU times: user 165 ms, sys: 2.97 ms, total: 168 ms
Wall time: 168 ms

alpha_beta vs. heuristic4


{'x': 0, 'o': 0, 'd': 1}

CPU times: user 944 ms, sys: 0 ns, total: 944 ms
Wall time: 944 ms


### Heuristic vs. Heuristic

In [14]:
DEBUG = 0

# Note: No randomness -> play only once

print("heuristic2 vs. heuristic4")
%time display(play(heuristic2_player, heuristic4_player, N = 1))

print()
print("heuristic4 vs. heuristic2")
%time display(play(heuristic4_player, heuristic2_player, N = 1))

heuristic2 vs. heuristic4


{'x': 0, 'o': 0, 'd': 1}

CPU times: user 28.1 ms, sys: 0 ns, total: 28.1 ms
Wall time: 27.8 ms

heuristic4 vs. heuristic2


{'x': 0, 'o': 0, 'd': 1}

CPU times: user 52.9 ms, sys: 996 μs, total: 53.9 ms
Wall time: 53.6 ms


## Cutoff comparison on the five test boards

This experiment compares the exact alpha-beta search (`cutoff=None`) with heuristic cutoffs at depths 2 and 4 on the five boards above. Depth counts plies from the current board (the root is depth 0). Each configuration is run three times; the reported time is the median. Node counts are deterministic for a given board and move order. Cell indices are numbered 0-8 from left to right, top to bottom. A value of `+1` is an X win, `0` a draw, and `-1` an X loss.

| Test board | Search | Cutoff | Chosen move | Value | Nodes |
|---|---|---:|---:|---:|---:|
| X about to win | Alpha-Beta full | - | 2 | +1 | 61 |
| X about to win | Heuristic Alpha-Beta | 2 | 8 | +1 | 13 |
| X about to win | Heuristic Alpha-Beta | 4 | 2 | +1 | 47 |
| O about to win | Alpha-Beta full | - | 2 | -1 | 15 |
| O about to win | Heuristic Alpha-Beta | 2 | 2 | -1 | 11 |
| O about to win | Heuristic Alpha-Beta | 4 | 2 | -1 | 15 |
| X can draw | Alpha-Beta full | - | 7 | 0 | 101 |
| X can draw | Heuristic Alpha-Beta | 2 | 7 | -0.4 | 21 |
| X can draw | Heuristic Alpha-Beta | 4 | 7 | 0 | 81 |
| Empty board | Alpha-Beta full | - | 0 | 0 | 18,297 |
| Empty board | Heuristic Alpha-Beta | 2 | 0 | 0 | 26 |
| Empty board | Heuristic Alpha-Beta | 4 | 4 | 0 | 541 |
| Bad situation | Alpha-Beta full | - | 1 | -1 | 238 |
| Bad situation | Heuristic Alpha-Beta | 2 | 4 | -0.8 | 29 |
| Bad situation | Heuristic Alpha-Beta | 4 | 1 | -1 | 148 |

Across the five boards, the search examined **18,712 nodes** for full alpha-beta, **100 nodes** at cutoff 2, and **832 nodes** at cutoff 4. The timings are printed by the executable cell immediately below; they vary by machine and load. The static comparison table intentionally lists only deterministic values to avoid stale benchmark numbers.

The deeper cutoff generally agrees more closely with full search: at cutoff 4, all five moves match full search except the empty board (center 4 instead of corner 0), while both choices still guarantee a draw. Cutoff 2 also finds immediate winning moves and the drawing move in the tactical cases, but its heuristic value on the draw board is -0.4 and it chooses a different move in the bad position. On the empty board, the cutoffs reduce the node count dramatically, but the improvement in speed comes at the cost of less complete tactical information.

In [15]:
import statistics
import time

comparison_boards = {
    'X about to win': {0: 'x', 1: 'o', 3: 'o', 4: 'x'},
    'O about to win': {0: 'o', 1: 'o', 3: 'o', 4: 'x', 8: 'x'},
    'X can draw': {0: 'x', 1: 'o', 2: 'x', 4: 'o'},
    'Empty board': {},
    'Bad situation': {0: 'o', 2: 'x', 8: 'o'},
}
cutoffs = [('Alpha-Beta full', None), ('Heuristic Alpha-Beta', 2), ('Heuristic Alpha-Beta', 4)]
repeats = 3
comparison_results = []
DEBUG = 0

for board_name, placements in comparison_boards.items():
    board = empty_board()
    for index, mark in placements.items():
        board[index] = mark
    for method, cutoff in cutoffs:
        elapsed_ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            search_result = alpha_beta_search(board, cutoff=cutoff)
            elapsed_ms.append((time.perf_counter() - start) * 1000)
        comparison_results.append({
            'board': board_name,
            'method': method,
            'cutoff': '-' if cutoff is None else cutoff,
            'move': search_result['move'],
            'value': search_result['value'],
            'nodes': COUNT,
            'median_ms': statistics.median(elapsed_ms),
        })

print(f'{"Board":<20} {"Method":<23} {"Cutoff":>6} {"Move":>5} {"Value":>7} {"Nodes":>8} {"Median ms":>12}')
for row in comparison_results:
    print(f"{row['board']:<20} {row['method']:<23} {str(row['cutoff']):>6} {str(row['move']):>5} {row['value']:>7} {row['nodes']:>8} {row['median_ms']:>12.3f}")

Board                Method                  Cutoff  Move   Value    Nodes    Median ms
X about to win       Alpha-Beta full              -     2       1       61        2.885
X about to win       Heuristic Alpha-Beta         2     8       1       13        0.715
X about to win       Heuristic Alpha-Beta         4     2       1       47        2.486
O about to win       Alpha-Beta full              -     2      -1       15        0.614
O about to win       Heuristic Alpha-Beta         2     2      -1       11        0.478
O about to win       Heuristic Alpha-Beta         4     2      -1       15        0.611
X can draw           Alpha-Beta full              -     7       0      101        4.623
X can draw           Heuristic Alpha-Beta         2     7    -0.4       21        1.107
X can draw           Heuristic Alpha-Beta         4     7       0       81        4.095
Empty board          Alpha-Beta full              -     0       0    18297      847.981
Empty board          Heuristic A

__Idea:__ Start experiments with different boards that already have a few x's and o's randomly placed on them.

### Discussion 3 — Khi nào cần Heuristic và Cutoff? (Hồ Minh Tiến)

Khi không gian tìm kiếm quá lớn để duyệt hết cây trong thời gian chấp nhận được, ta giới hạn độ sâu bằng `cutoff` và dùng heuristic để ước lượng các trạng thái chưa kết thúc. Trong notebook này, `alpha_beta_search(board, cutoff=None)` là tìm kiếm đầy đủ; `cutoff=2` và `cutoff=4` là hai mức cắt được so sánh trên cùng năm thế cờ.

Cutoff nhỏ thường giảm đáng kể số nút và thời gian nhưng không bảo đảm luôn chọn đúng nước tối ưu; cutoff lớn có thể nhận ra nhiều chiến thuật hơn, đổi lại chi phí cao hơn. Kết quả trên bàn trống và các bàn chiến thuật trong bảng phía trên chứng minh đó là **đánh đổi giữa chi phí và độ chính xác**, không phải luôn tăng cutoff là sẽ chọn nước khác hay mạnh hơn. Ngoài ra, trên Connect 4 chuẩn, phần Task 3 đã vượt 500.000 nút khi tăng từ 4×4 lên 4×5, còn Task 4 của Tiến dùng heuristic cửa sổ bốn ô và cắt độ sâu để tìm kiếm trên 6×7.

Ô bên dưới tự tổng hợp số nút và số nước trùng với Alpha-Beta đầy đủ từ bảng thực nghiệm mới chạy. Đây là mức khớp **nước đi**, không thay thế đánh giá sức mạnh qua nhiều ván.

In [16]:
# Tổng hợp có thể tái lập cho Discussion 3 từ bảng thực nghiệm ngay phía trên.
case_names = list(comparison_boards)
full_search = {row['board']: row for row in comparison_results
               if row['method'] == 'Alpha-Beta full'}
for level in (2, 4):
    selected = {row['board']: row for row in comparison_results
                if row['method'] == 'Heuristic Alpha-Beta' and row['cutoff'] == level}
    matched = sum(selected[name]['move'] == full_search[name]['move'] for name in case_names)
    nodes = sum(selected[name]['nodes'] for name in case_names)
    elapsed = sum(selected[name]['median_ms'] for name in case_names)
    print(f'Cutoff {level}: matches full AB = {matched}/{len(case_names)}; '
          f'visited nodes = {nodes}; sum of per-case median times = {elapsed:.1f} ms')
full_nodes = sum(row['nodes'] for row in full_search.values())
print(f'Full Alpha-Beta: visited nodes = {full_nodes}; cases = {len(case_names)}')
assert len(comparison_results) == 3 * len(case_names)
assert full_nodes >= sum(row['nodes'] for row in comparison_results
                         if row['method'] == 'Heuristic Alpha-Beta' and row['cutoff'] == 4)

Cutoff 2: matches full AB = 3/5; visited nodes = 100; sum of per-case median times = 5.4 ms
Cutoff 4: matches full AB = 4/5; visited nodes = 832; sum of per-case median times = 45.8 ms
Full Alpha-Beta: visited nodes = 18712; cases = 5
